# **Loading Packages**
_____

In [1]:
import pandas as pd
import numpy as np

import requests
import time
from io import StringIO
import os

from fredapi import Fred
import yfinance as yf

from sklearn.linear_model import LassoCV
from statsmodels.tsa.stattools import adfuller

# **Calling Data**
______

## **FRED**

In [2]:
os.chdir('/Users/dannyhogan/Desktop/ST-498/Code')
with open("API_key.txt", "r") as file:
    api_key = file.read().strip()
fred = Fred(api_key=api_key)

# --- UK FRED Series ---
UK_exchange_rate = fred.get_series('RBGBBIS').resample('QS').first()
UK_oil_price     = fred.get_series('DCOILBRENTEU').resample('QS').first()
UK_bond_yield    = fred.get_series('IRLTLT01GBM156N').resample('QS').first()
UK_GDP           = fred.get_series('NGDPRSAXDCGBQ')
UK_Credit        = fred.get_series('QGBCAMUSDA')

Fred_Data_UK = pd.DataFrame({
    'UK_ExchangeRate': UK_exchange_rate,
    'UK_OilPrice':     UK_oil_price,
    'UK_BondYield':    UK_bond_yield,
    'UK_rGDP':         UK_GDP,
    'UK_Credit':       UK_Credit.pct_change(4) * 100
})

Fred_Data_UK = Fred_Data_UK[Fred_Data_UK.index > '1993-10-01']

# --- US FRED Series ---
US_exchange_rate = fred.get_series('DEXUSEU').resample('QS').first()         # USD/EUR exchange rate
US_oil_price     = fred.get_series('DCOILWTICO').resample('QS').first()      # WTI crude oil
US_bond_yield    = fred.get_series('IRLTLT01USM156N').resample('QS').first() # US 10-year bond yield
US_GDP           = fred.get_series('GDPC1')                                   # US Real GDP (quarterly)
US_Credit        = fred.get_series('DRCCLACBS').resample('QS').first()       # Delinquency Rate on Credit Card Loans

Fred_Data_US = pd.DataFrame({
    'US_ExchangeRate': US_exchange_rate,
    'US_OilPrice':     US_oil_price,
    'US_BondYield':    US_bond_yield,
    'US_rGDP':         US_GDP,
    'US_Credit':       US_Credit  # Already a rate, no pct_change needed
})

Fred_Data_US = Fred_Data_US[Fred_Data_US.index > '1993-10-01']

## **Yahoo Finance**

In [3]:
# --- UK: FTSE 100 ---
ftse = yf.Ticker("^FTSE")
ftse_data = ftse.history(
    start="1993-01-01",
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"
)
ftse_data = ftse_data[['Open']].rename(columns={'Open': 'UK_FTSE_Open'})

# --- US: S&P 500 ---
sp500 = yf.Ticker("^GSPC")
sp500_data = sp500.history(
    start="1993-01-01",
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"
)
sp500_data = sp500_data[['Open']].rename(columns={'Open': 'US_SP500_Open'})

## **OECD**

In [17]:
def fetch_oecd(url, name):
    """Fetch OECD data and return cleaned DataFrame"""
    if 'format=' not in url:
        url += '&format=csvfilewithlabels'
    
    headers = {
        'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    time.sleep(1)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
        df_clean.columns = ['date', name]
        df_clean[name] = pd.to_numeric(df_clean[name], errors='coerce')
        return df_clean
    else:
        print(f"{name}: Error {response.status_code}")
        return None

# ======================================================================
# Update URLs here
# ======================================================================

# --- UK OECD Queries ---
queries_UK = {
    "UK_house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/GBR.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "UK_cpi":          "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/GBR.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "UK_unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/GBR..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

# --- US OECD Queries ---
queries_US = {
    "US_house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/USA.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "US_cpi":          "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/USA.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "US_unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/USA..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

dfs_UK = [fetch_oecd(url, name) for name, url in queries_UK.items()]
OECDdata_UK = pd.concat([df.set_index('date') for df in dfs_UK if df is not None], axis=1)

dfs_US = [fetch_oecd(url, name) for name, url in queries_US.items()]
OECDdata_US = pd.concat([df.set_index('date') for df in dfs_US if df is not None], axis=1)

In [18]:
OECDdata_UK

,UK_house_prices,UK_cpi,UK_unemployment
date,,,
1989-Q2,54.135673,5.266667,7.1
1989-Q3,54.293299,5.133333,7.0
1990-Q3,49.250125,7.533333,6.8
1990-Q4,47.990513,7.833333,7.1
1991-Q3,44.559421,7.766667,9.0
...,...,...,...
2020-Q2,110.293465,0.633333,4.1
2022-Q1,120.104977,6.233333,3.8
2022-Q4,116.902087,10.766670,3.9


# **Creating Dataset**
___

In [20]:
# --- UK Dataset ---
OECDdata_UK.index = pd.PeriodIndex(OECDdata_UK.index, freq='Q').to_timestamp()

Fred_Data_UK.index = pd.to_datetime(Fred_Data_UK.index).tz_localize(None)
ftse_data.index    = pd.to_datetime(ftse_data.index).tz_localize(None)

merged_UK = OECDdata_UK.join([Fred_Data_UK, ftse_data], how='outer')
merged_UK = merged_UK[merged_UK.index > '1993-12-31']

# --- US Dataset ---
OECDdata_US.index = pd.PeriodIndex(OECDdata_US.index, freq='Q').to_timestamp()

Fred_Data_US.index = pd.to_datetime(Fred_Data_US.index).tz_localize(None)
sp500_data.index   = pd.to_datetime(sp500_data.index).tz_localize(None)

merged_US = OECDdata_US.join([Fred_Data_US, sp500_data], how='outer')
merged_US = merged_US[merged_US.index > '1993-12-31']

# **Adding Lag Variables for Regression Models:**
______

In [22]:
def check_stationarity(series, name):
    result = adfuller(series.dropna())
    pval = result[1]
    print(f"{name:25s} | ADF p-value: {pval:.4f} | {'Stationary ✓' if pval < 0.05 else 'Non-stationary ✗'}")

def build_lag_matrix(df, target, max_lag=4):
    X = pd.DataFrame()
    for col in df.columns:
        if col == target:
            continue
        for lag in range(2, max_lag + 1):   # lag 2+ only (predictive model)
            X[f'{col}_L{lag}'] = df[col].shift(lag)
    y = df[target]
    X = X.dropna()
    y = y[y.index.isin(X.index)]
    return X, y

def run_lag_analysis(df, country):
    """
    Runs stationarity checks, transforms, and LassoCV lag selection
    for a given country dataframe. Returns surviving lag coefficients.
    """
    credit_col = f'{country}_Credit'

    # ── column sets ───────────────────────────────────────────────────────────
    log_diff_cols = [c for c in ['rGDP', 'house_prices', 'OilPrice', 'ExchangeRate', 'Open',
                                  'UK_rGDP', 'US_rGDP',
                                  'UK_house_prices', 'US_house_prices',
                                  'UK_OilPrice', 'US_OilPrice',
                                  'UK_ExchangeRate', 'US_ExchangeRate',
                                  'UK_FTSE_Open', 'US_SP500_Open'] if c in df.columns]

    first_diff_cols = [c for c in df.columns
                       if c not in log_diff_cols and c != credit_col]

    print(f"\n{'='*60}")
    print(f"  {country} — Stationarity Checks")
    print(f"{'='*60}")
    for col in df.columns:
        check_stationarity(df[col], col)

    # ── transform ─────────────────────────────────────────────────────────────
    df_transformed = pd.DataFrame()

    for col in log_diff_cols:
        df_transformed[f'd_{col}'] = np.log(df[col]).diff()

    for col in first_diff_cols:
        df_transformed[f'd_{col}'] = df[col].diff()

    # credit target — first difference
    target_name = f'd_{credit_col}'
    df_transformed[target_name] = df[credit_col].diff()

    df_transformed.dropna(inplace=True)

    # ── LassoCV ───────────────────────────────────────────────────────────────
    X, y = build_lag_matrix(df_transformed, target_name, max_lag=4)

    lasso = LassoCV(cv=5, max_iter=10000)
    lasso.fit(X, y)

    important = pd.Series(lasso.coef_, index=X.columns)
    surviving = important[important != 0].sort_values()

    print(f"\n  {country} — Surviving Lags (L2+) for {target_name}:")
    print(surviving if not surviving.empty else "  None survived regularisation.")

    return surviving

# ── run ───────────────────────────────────────────────────────────────────────

surviving_UK = run_lag_analysis(merged_UK, 'UK')
surviving_US = run_lag_analysis(merged_US, 'US')


  UK — Stationarity Checks
UK_house_prices           | ADF p-value: 0.4047 | Non-stationary ✗
UK_cpi                    | ADF p-value: 0.0037 | Stationary ✓
UK_unemployment           | ADF p-value: 0.1760 | Non-stationary ✗
UK_ExchangeRate           | ADF p-value: 0.5384 | Non-stationary ✗
UK_OilPrice               | ADF p-value: 0.1054 | Non-stationary ✗
UK_BondYield              | ADF p-value: 0.5802 | Non-stationary ✗
UK_rGDP                   | ADF p-value: 0.6995 | Non-stationary ✗
UK_Credit                 | ADF p-value: 0.1062 | Non-stationary ✗
UK_FTSE_Open              | ADF p-value: 0.7992 | Non-stationary ✗

  UK — Surviving Lags (L2+) for d_UK_Credit:
d_UK_cpi_L2            -1.275325
d_UK_OilPrice_L3       -0.065456
d_UK_cpi_L4             1.764517
d_UK_unemployment_L2    4.598637
dtype: float64

  US — Stationarity Checks
US_house_prices           | ADF p-value: 0.7094 | Non-stationary ✗
US_cpi                    | ADF p-value: 0.0427 | Stationary ✓
US_unemployment       

# **Saving Data Locally**
_____

In [23]:
df_UK_final = merged_UK.join(surviving_UK.rename('UK_Credit_lags'), how='left')
df_US_final = merged_US.join(surviving_US.rename('US_Credit_lags'), how='left')

In [24]:
# Be sure to change to your own internal directory
os.chdir('/Users/dannyhogan/Desktop/ST-498')
df_UK_final.to_csv("Data/MacroVariablesUK.csv")
df_US_final.to_csv("Data/MacroVariablesUS.csv")